# Step 3 — TensorFlow/Keras Multimodal Survival Model

Architecture: Static valve MLP + Physiology GRU + Medication GRU + Fusion + discrete-time survival head.

In [12]:
from pathlib import Path
import json, numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED=42
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_CSV = Path("synthetic_generator_outputs_v2/synthetic_multimodal_patient_year_v2_MODEL_READY.csv")
SPEC_JSON = Path("synthetic_generator_outputs_v2/encoder_input_spec_v2.json")

print("TensorFlow:", tf.__version__)
print("Data:", DATA_CSV.resolve())
print("Spec:", SPEC_JSON.resolve())

TensorFlow: 2.10.0
Data: C:\Users\User\OneDrive\Υπολογιστής\πανεπιστημιο\workstation\docathon\synthetic_generator_outputs_v2\synthetic_multimodal_patient_year_v2_MODEL_READY.csv
Spec: C:\Users\User\OneDrive\Υπολογιστής\πανεπιστημιο\workstation\docathon\synthetic_generator_outputs_v2\encoder_input_spec_v2.json


In [13]:
df = pd.read_csv(DATA_CSV)
with open(SPEC_JSON, "r", encoding="utf-8") as f:
    spec = json.load(f)

PATIENT_COL = spec["patient_id_column"]
TIME_COL = spec["time_column"]
STATIC_FEATURES = [c for c in spec["static_features"] if c != "valve_position"]
PHYS_FEATURES = spec["physiology_features"]
MED_FEATURES = spec["medication_features"]
NOTE_FEATURES = spec.get("note_derived_features_v1_optional", [])

print(df.shape, df[PATIENT_COL].nunique())

(7169, 67) 1000


In [14]:
patient_table = df[[PATIENT_COL,"event","duration_months"]].drop_duplicates(PATIENT_COL).reset_index(drop=True)

g1 = GroupShuffleSplit(n_splits=1, train_size=0.70, random_state=SEED)
tr_idx, tmp_idx = next(g1.split(patient_table, groups=patient_table[PATIENT_COL]))
train_patients = patient_table.iloc[tr_idx][PATIENT_COL].tolist()
tmp = patient_table.iloc[tmp_idx].reset_index(drop=True)

g2 = GroupShuffleSplit(n_splits=1, train_size=0.50, random_state=SEED+1)
va_idx, te_idx = next(g2.split(tmp, groups=tmp[PATIENT_COL]))
val_patients = tmp.iloc[va_idx][PATIENT_COL].tolist()
test_patients = tmp.iloc[te_idx][PATIENT_COL].tolist()

print(len(train_patients), len(val_patients), len(test_patients))

700 150 150


In [15]:
train_df = df[df[PATIENT_COL].isin(train_patients)].copy()

STATIC_NUMERIC = [c for c in STATIC_FEATURES if pd.api.types.is_numeric_dtype(train_df[c])]
STATIC_CATEGORICAL = [c for c in STATIC_FEATURES if c not in STATIC_NUMERIC]

static_rows = train_df.sort_values([PATIENT_COL,TIME_COL]).groupby(PATIENT_COL).first().reset_index()

static_scaler = StandardScaler().fit(static_rows[STATIC_NUMERIC].astype(float))
try:
    static_ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    static_ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)
static_ohe.fit(static_rows[STATIC_CATEGORICAL].astype(str))

phys_train = train_df[PHYS_FEATURES].apply(pd.to_numeric, errors="coerce")
phys_mean = phys_train.mean().fillna(0.0)
phys_std = phys_train.std().replace(0,1.0).fillna(1.0)
TIME_SCALE = max(float(train_df[TIME_COL].max()),1.0)

In [16]:
def transform_static(pdf):
    first = pdf.sort_values(TIME_COL).iloc[0]
    num = static_scaler.transform(first[STATIC_NUMERIC].astype(float).to_numpy().reshape(1,-1))
    cat = static_ohe.transform(first[STATIC_CATEGORICAL].astype(str).to_numpy().reshape(1,-1))
    return np.concatenate([num,cat],axis=1).astype(np.float32).squeeze(0)

def transform_phys(pdf):
    pdf = pdf.sort_values(TIME_COL)
    vals = pdf[PHYS_FEATURES].apply(pd.to_numeric, errors="coerce")
    mask = vals.notna().astype(np.float32)
    z = ((vals-phys_mean)/phys_std).fillna(0.0).astype(np.float32)
    parts = [z.to_numpy(), mask.to_numpy()]
    if NOTE_FEATURES:
        parts.append(pdf[NOTE_FEATURES].apply(pd.to_numeric,errors="coerce").fillna(0.0).astype(np.float32).to_numpy())
    t = (pdf[TIME_COL].astype(float).to_numpy().reshape(-1,1)/TIME_SCALE).astype(np.float32)
    parts.append(t)
    return np.concatenate(parts,axis=1).astype(np.float32)

def transform_med(pdf):
    pdf = pdf.sort_values(TIME_COL)
    vals = pdf[MED_FEATURES].apply(pd.to_numeric, errors="coerce")
    mask = vals.notna().astype(np.float32)
    filled = vals.fillna(0.0).astype(np.float32)
    t = (pdf[TIME_COL].astype(float).to_numpy().reshape(-1,1)/TIME_SCALE).astype(np.float32)
    return np.concatenate([filled.to_numpy(),mask.to_numpy(),t],axis=1).astype(np.float32)

def make_sample(pdf):
    pdf = pdf.sort_values(TIME_COL)
    return {
        "pid": str(pdf[PATIENT_COL].iloc[0]),
        "xs": transform_static(pdf),
        "xp": transform_phys(pdf),
        "xm": transform_med(pdf),
        "duration": float(pdf["duration_months"].iloc[0]),
        "event": float(pdf["event"].iloc[0]),
    }

In [17]:
def build_arrays(ids):
    samples=[make_sample(df[df[PATIENT_COL]==pid]) for pid in ids]
    max_t=max(s["xp"].shape[0] for s in samples)
    sd=samples[0]["xs"].shape[0]; pdim=samples[0]["xp"].shape[1]; mdim=samples[0]["xm"].shape[1]
    xs=np.zeros((len(samples),sd),np.float32)
    xp=np.zeros((len(samples),max_t,pdim),np.float32)
    xm=np.zeros((len(samples),max_t,mdim),np.float32)
    dur=np.zeros(len(samples),np.float32); ev=np.zeros(len(samples),np.float32)
    pids=[]
    for i,s in enumerate(samples):
        t=s["xp"].shape[0]
        xs[i]=s["xs"]; xp[i,:t]=s["xp"]; xm[i,:t]=s["xm"]
        dur[i]=s["duration"]; ev[i]=s["event"]; pids.append(s["pid"])
    return {"pid":np.array(pids),"xs":xs,"xp":xp,"xm":xm,"duration":dur,"event":ev}

train=build_arrays(train_patients)
val=build_arrays(val_patients)
test=build_arrays(test_patients)

print(train["xs"].shape, train["xp"].shape, train["xm"].shape)

c:\Users\User\miniconda3\envs\tf_directml\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\User\miniconda3\envs\tf_directml\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\User\miniconda3\envs\tf_directml\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\User\miniconda3\envs\tf_directml\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\User\miniconda3\envs\tf_directml\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler

(700, 13) (700, 11, 71) (700, 11, 31)


c:\Users\User\miniconda3\envs\tf_directml\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\User\miniconda3\envs\tf_directml\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\User\miniconda3\envs\tf_directml\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\User\miniconda3\envs\tf_directml\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\User\miniconda3\envs\tf_directml\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler

In [18]:
BIN_EDGES=np.arange(0,121,12,dtype=np.float32)
N_BINS=len(BIN_EDGES)-1

def make_targets(duration,event):
    y_event=np.zeros((len(duration),N_BINS),np.float32)
    y_risk=np.zeros((len(duration),N_BINS),np.float32)
    for i,(d,e) in enumerate(zip(duration,event)):
        k=min(max(int(np.floor(d/12.0)),0),N_BINS-1)
        y_risk[i,:k+1]=1.0
        if e>0.5: y_event[i,k]=1.0
    return np.concatenate([y_event,y_risk],axis=1).astype(np.float32)

y_train=make_targets(train["duration"],train["event"])
y_val=make_targets(val["duration"],val["event"])
y_test=make_targets(test["duration"],test["event"])

def survival_loss(y_true,y_pred):
    y_event=y_true[:,:N_BINS]
    y_risk=y_true[:,N_BINS:]
    h=tf.clip_by_value(y_pred,1e-7,1-1e-7)
    ll=y_event*tf.math.log(h)+(y_risk-y_event)*tf.math.log(1-h)
    return tf.reduce_mean(-tf.reduce_sum(ll,axis=1))

In [19]:
STATIC_DIM=train["xs"].shape[1]
PHYS_DIM=train["xp"].shape[2]
MED_DIM=train["xm"].shape[2]

static_in=keras.Input((STATIC_DIM,),name="static_input")
phys_in=keras.Input((None,PHYS_DIM),name="phys_input")
med_in=keras.Input((None,MED_DIM),name="med_input")

x=layers.Dense(64,activation="relu")(static_in)
x=layers.Dropout(0.15)(x)
z_valve=layers.Dense(32,activation="relu",name="z_valve")(x)

p=layers.Masking(mask_value=0.0)(phys_in)
p=layers.GRU(64, reset_after=False, name="phys_gru")(p)
z_phys=layers.Dense(32,activation="relu",name="z_phys")(p)

m=layers.Masking(mask_value=0.0)(med_in)
m=layers.GRU(48, reset_after=False, name="med_gru")(m)
z_med=layers.Dense(32,activation="relu",name="z_med")(m)

temporal=layers.Concatenate()([z_phys,z_med])
temporal=layers.Dense(64,activation="relu")(temporal)
z_patient=layers.Dense(64,activation="relu",name="z_patient")(temporal)

fused=layers.Concatenate()([z_patient,z_valve])
fused=layers.Dense(64,activation="relu")(fused)
hazards=layers.Dense(N_BINS,activation="sigmoid",name="hazards")(fused)

model=keras.Model([static_in,phys_in,med_in],hazards)
model.compile(optimizer=keras.optimizers.Adam(1e-3),loss=survival_loss)
model.summary()

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 phys_input (InputLayer)        [(None, None, 71)]   0           []                               
                                                                                                  
 med_input (InputLayer)         [(None, None, 31)]   0           []                               
                                                                                                  
 masking_2 (Masking)            (None, None, 71)     0           ['phys_input[0][0]']             
                                                                                                  
 masking_3 (Masking)            (None, None, 31)     0           ['med_input[0][0]']              
                                                                                            

## DirectML compatibility fix

On some Windows TensorFlow-DirectML environments, Keras GRU tries to use `CudnnRNNV3`, which DirectML does not provide. In this notebook the GRUs are created with `reset_after=False`, forcing the standard TensorFlow GRU path instead of the cuDNN kernel.

**Restart the notebook kernel and Run All** so the old GRU graph is not reused.

In [20]:
# callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss",patience=6,restore_best_weights=True)]

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        min_delta=1e-4
    )
]

history=model.fit(
    {"static_input":train["xs"],"phys_input":train["xp"],"med_input":train["xm"]},
    y_train,
    validation_data=(
        {"static_input":val["xs"],"phys_input":val["xp"],"med_input":val["xm"]},
        y_val
    ),
    epochs=30,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/30
11/11 [==============================] - 4s 130ms/step - loss: 3.7722 - val_loss: 2.4886
Epoch 2/30
11/11 [==============================] - 1s 91ms/step - loss: 1.5416 - val_loss: 0.8943
Epoch 3/30
11/11 [==============================] - 1s 86ms/step - loss: 1.0366 - val_loss: 0.8549
Epoch 4/30
11/11 [==============================] - 1s 89ms/step - loss: 0.9054 - val_loss: 0.8062
Epoch 5/30
11/11 [==============================] - 1s 85ms/step - loss: 0.8747 - val_loss: 0.7636
Epoch 6/30
11/11 [==============================] - 1s 86ms/step - loss: 0.8432 - val_loss: 0.7534
Epoch 7/30
11/11 [==============================] - 1s 87ms/step - loss: 0.8139 - val_loss: 0.7477
Epoch 8/30
11/11 [==============================] - 1s 90ms/step - loss: 0.7832 - val_loss: 0.7262
Epoch 9/30
11/11 [==============================] - 1s 92ms/step - loss: 0.7582 - val_loss: 0.7103
Epoch 10/30
11/11 [==============================] - 1s 89ms/step - loss: 0.7208 - val_loss: 0.7029
Epoch 11

In [21]:
test_loss=model.evaluate(
    {"static_input":test["xs"],"phys_input":test["xp"],"med_input":test["xm"]},
    y_test, verbose=0
)
print("Test survival NLL:",float(test_loss))

haz=model.predict(
    {"static_input":test["xs"],"phys_input":test["xp"],"med_input":test["xm"]},
    verbose=0
)
surv=np.cumprod(1-haz+1e-7,axis=1)
print("Example S(t):",surv[0])

assert np.all(np.diff(surv,axis=1)<=1e-6)

Test survival NLL: 0.5795695781707764
Example S(t): [0.9768956  0.9244934  0.83577895 0.65941167 0.36831313 0.25515348
 0.17857634 0.09614598 0.06766944 0.04981242]


In [22]:
OUT=Path("model_outputs_tensorflow_v1")
OUT.mkdir(parents=True,exist_ok=True)

model.save(OUT/"multimodal_valve_survival_model_v1.keras")

rows=[]
for i,pid in enumerate(test["pid"]):
    r={"Patient":pid,"duration_months":float(test["duration"][i]),"event":int(test["event"][i])}
    for k in range(N_BINS):
        month=int(BIN_EDGES[k+1])
        r[f"hazard_{month}m"]=float(haz[i,k])
        r[f"S_{month}m"]=float(surv[i,k])
    rows.append(r)

pd.DataFrame(rows).to_csv(OUT/"test_survival_predictions_v1.csv",index=False)
pd.DataFrame(history.history).to_csv(OUT/"training_history_v1.csv",index=False)

print("Saved to:",OUT.resolve())

Saved to: C:\Users\User\OneDrive\Υπολογιστής\πανεπιστημιο\workstation\docathon\model_outputs_tensorflow_v1


## Next step

After this runs, we evaluate C-index, Brier score, calibration and modality ablations before making the architecture more complex.